# SQL reference 10: changing data: DML, MERGE and history

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-10-changing-data.ipynb)

CTAS, `INSERT` and partition replacement, `UPDATE` and `DELETE`, `MERGE` upserts with de-duplicated sources and late-data guards, full-snapshot synchronisation, SCD type 2 in one `MERGE`, and Delta time travel. Practise it in [SQL practice 10: changing data: DML, MERGE and history](sql-practice-10-changing-data.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose. This notebook uses a Delta Lake session (the Delta jars download on first use), because `UPDATE`, `DELETE` and `MERGE` need a table format that supports them. `fresh(table, select)` drops and re-creates a Delta table.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="sql-")  # tables created in this notebook live here

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, UPDATE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

def fresh(table, select):
    """Drop and re-create a Delta table from a query, so its history starts again at version 0."""
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    spark.sql(f"CREATE TABLE {table} USING DELTA AS {select}")

## Contents

| section | entries |
| --- | --- |
| [Creating and loading tables](#creating-and-loading-tables) | [`CREATE TABLE AS SELECT` and `INSERT`](#create-table-as-select-and-insert) |
| [Modifying rows](#modifying-rows) | [`UPDATE` and `DELETE`](#update-and-delete) |
| [MERGE](#merge) | [`MERGE INTO` (upsert)](#merge-into-upsert), [Synchronising a table with a full snapshot](#synchronising-a-table-with-a-full-snapshot) |
| [History](#history) | [Slowly changing dimensions type 2 with `MERGE`](#slowly-changing-dimensions-type-2-with-merge), [Transactions, time travel and restore](#transactions-time-travel-and-restore) |

## Creating and loading tables

### `CREATE TABLE AS SELECT` and `INSERT`

| statement | does |
| --- | --- |
| `CREATE [OR REPLACE] TABLE t USING DELTA AS SELECT ...` (CTAS) | create a table with the query's schema and rows |
| `INSERT INTO t [(cols)] VALUES (...), (...)` | append literal rows |
| `INSERT INTO t SELECT ...` | append a query result (columns matched **by position** unless you list them, or use `BY NAME`) |
| `INSERT OVERWRITE t SELECT ...` | replace all rows |
| `INSERT INTO t REPLACE WHERE pred SELECT ...` (Delta) | atomically replace only the rows matching `pred` (one partition, one day) |

**Notes:** appends are not idempotent: re-running an `INSERT INTO` after a partial failure duplicates rows. Pipelines
prefer `MERGE` (upsert) or overwriting a whole partition, which give the same result however often they run.

In [ ]:
fresh("fact_orders", "SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount FROM orders WHERE order_id <= 6")
spark.sql("INSERT INTO fact_orders VALUES (100, 1, 'placed', DATE'2026-01-20', 42.00)")
spark.sql("INSERT INTO fact_orders BY NAME SELECT 101 AS order_id, 2 AS customer_id, DATE'2026-01-20' AS order_date, 'paid' AS status, 10.00 AS amount")
show("SELECT order_date, count(*) AS orders, sum(amount) AS revenue FROM fact_orders GROUP BY order_date ORDER BY order_date")
spark.sql("""
INSERT INTO fact_orders REPLACE WHERE order_date = DATE'2026-01-20'
SELECT 100 AS order_id, 1 AS customer_id, 'paid' AS status, DATE'2026-01-20' AS order_date, 45.00 AS amount
""")
show("SELECT * FROM fact_orders WHERE order_date = DATE'2026-01-20'")

## Modifying rows

### `UPDATE` and `DELETE`

```sql
UPDATE t SET col = expr [, ...] [WHERE pred]
DELETE FROM t [WHERE pred]
```

| detail | behaviour |
| --- | --- |
| table support | Delta, Iceberg and Hudi tables; plain Parquet/CSV tables and views cannot be updated |
| subqueries | Delta does not allow subqueries in `UPDATE`/`DELETE` conditions (`DELTA_UNSUPPORTED_SUBQUERY`): delete or update via `MERGE ... WHEN MATCHED THEN DELETE` instead |
| atomicity | each statement is one transaction: all matching rows change, or none |
| cost | Delta rewrites every **file** that contains a matching row (or marks rows deleted with deletion vectors) |

**Notes:** run the `WHERE` as a `SELECT count(*)` first; an `UPDATE` or `DELETE` without `WHERE` changes every row.

In [ ]:
fresh("fact_orders", "SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount FROM orders")
spark.sql("UPDATE fact_orders SET status = 'shipped' WHERE status = 'paid' AND order_date < DATE'2026-01-08'")
spark.sql("DELETE FROM fact_orders WHERE status = 'returned'")
spark.sql("""
MERGE INTO fact_orders t
USING (SELECT customer_id FROM customers WHERE region IS NULL) s    -- "DELETE ... WHERE key IN (subquery)" in Delta
ON t.customer_id = s.customer_id
WHEN MATCHED THEN DELETE
""")
show("SELECT status, count(*) AS orders FROM fact_orders GROUP BY status ORDER BY status")

## MERGE

### `MERGE INTO` (upsert)

```sql
MERGE INTO target t
USING source s
ON t.key = s.key
WHEN MATCHED [AND cond] THEN UPDATE SET ... | UPDATE SET * | DELETE
WHEN NOT MATCHED [BY TARGET] [AND cond] THEN INSERT (...) VALUES (...) | INSERT *
WHEN NOT MATCHED BY SOURCE [AND cond] THEN UPDATE SET ... | DELETE
```

| clause | applies to |
| --- | --- |
| `WHEN MATCHED` | target rows with a matching source row |
| `WHEN NOT MATCHED` | source rows with no target row (new records) |
| `WHEN NOT MATCHED BY SOURCE` | target rows absent from the source (full synchronisation, soft deletes) |
| `UPDATE SET *` / `INSERT *` | all columns by name |

Clauses of the same kind are tried in order; the first whose condition holds wins.

**Notes:** each target row may match **at most one** source row, otherwise MERGE fails
(`DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`): de-duplicate the source first, typically keeping the latest
version per key with `row_number()`. Add `AND s.updated_at > t.updated_at` to `WHEN MATCHED` so late, older records
never overwrite newer ones. A MERGE with a deterministic, de-duplicated source is idempotent.

In [ ]:
fresh("dim_customer", "SELECT customer_id, name, region, DATE'2026-01-01' AS updated_at FROM customers WHERE customer_id <= 4")
spark.sql("""
CREATE OR REPLACE TEMP VIEW customer_changes AS SELECT * FROM VALUES
  (2, 'Bright Retail', 'South', DATE'2026-01-10'),    -- moved region
  (4, 'Delta Grocers', 'West',  DATE'2025-12-01'),    -- older than the target row: ignore
  (9, 'Iris Imports',  'North', DATE'2026-01-10'),    -- new customer
  (9, 'Iris Imports',  'East',  DATE'2026-01-12')     -- newer version of the same key in this batch
AS c(customer_id, name, region, updated_at)
""")
spark.sql("""
MERGE INTO dim_customer t
USING (
    SELECT * FROM (SELECT *, row_number() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn FROM customer_changes)
    WHERE rn = 1
) s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET name = s.name, region = s.region, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, name, region, updated_at) VALUES (s.customer_id, s.name, s.region, s.updated_at)
""")
show("SELECT * FROM dim_customer ORDER BY customer_id")
with expected_error():
    spark.sql("MERGE INTO dim_customer t USING customer_changes s ON t.customer_id = s.customer_id WHEN MATCHED THEN UPDATE SET *")

### Synchronising a table with a full snapshot

When the source is a complete snapshot, `WHEN NOT MATCHED BY SOURCE THEN DELETE` removes target rows that disappeared,
in the same transaction as the updates and inserts. Use `UPDATE SET is_active = false` instead of `DELETE` for soft
deletes.

In [ ]:
fresh("dim_product", "SELECT sku, category, list_price FROM products WHERE sku <= 'SKU-4004'")
spark.sql("""
MERGE INTO dim_product t
USING (SELECT * FROM VALUES ('SKU-4001', 'Kitchen', 13.00), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4099', 'Garden', 5.00)
       AS s(sku, category, list_price)) s
ON t.sku = s.sku
WHEN MATCHED AND (t.list_price <> s.list_price OR t.category <> s.category) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
show("SELECT * FROM dim_product ORDER BY sku")

## History

### Slowly changing dimensions type 2 with `MERGE`

A type 2 dimension keeps history: a change closes the current row (`valid_to`, `is_current = false`) and inserts a new
current row. One MERGE can do both by **staging** each changed record twice: once with its real key (to match and close
the current row) and once with a NULL merge key (which never matches, so it is inserted).

| staged row | merge key | effect |
| --- | --- | --- |
| changed record | the business key | `WHEN MATCHED AND changed` closes the current row |
| changed record, second copy | NULL | `WHEN NOT MATCHED` inserts the new version |
| brand-new record | the business key | `WHEN NOT MATCHED` inserts it |

In [ ]:
fresh("dim_customer_scd", """
SELECT customer_id, name, region, DATE'2025-01-01' AS valid_from, CAST(NULL AS DATE) AS valid_to, true AS is_current
FROM customers WHERE customer_id <= 3
""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW scd_changes AS SELECT * FROM VALUES
  (2, 'Bright Retail', 'South'), (3, 'Cyan Traders', 'South'), (9, 'Iris Imports', 'North')
AS c(customer_id, name, region)
""")
spark.sql("""
MERGE INTO dim_customer_scd t
USING (
    SELECT c.customer_id AS merge_key, c.* FROM scd_changes c
    UNION ALL
    SELECT NULL AS merge_key, c.* FROM scd_changes c
    JOIN dim_customer_scd d ON d.customer_id = c.customer_id AND d.is_current AND d.region <> c.region
) s
ON t.customer_id = s.merge_key AND t.is_current
WHEN MATCHED AND t.region <> s.region THEN UPDATE SET valid_to = DATE'2026-01-15', is_current = false
WHEN NOT MATCHED THEN INSERT (customer_id, name, region, valid_from, valid_to, is_current)
                      VALUES (s.customer_id, s.name, s.region, DATE'2026-01-15', NULL, true)
""")
show("SELECT * FROM dim_customer_scd ORDER BY customer_id, valid_from")

### Transactions, time travel and restore

Delta (like Iceberg and Hudi) records every committed change as a new table **version**:

| statement | does |
| --- | --- |
| `DESCRIBE HISTORY t` | versions, timestamps and operations |
| `SELECT ... FROM t VERSION AS OF n` / `TIMESTAMP AS OF '...'` | read an old version (audits, debugging, reproducing a report) |
| `RESTORE TABLE t TO VERSION AS OF n` | roll the table back (itself a new version) |
| `VACUUM t [RETAIN n HOURS]` | delete files no longer referenced, which limits how far back time travel works |

**Notes:** each statement is atomic and isolated (readers never see half a write), but Spark SQL has no multi-statement
`BEGIN ... COMMIT` transactions; design each load step as one idempotent statement. Warehouses such as Postgres,
Snowflake and SQL Server support multi-statement transactions.

In [ ]:
fresh("audit_demo", "SELECT order_id, status FROM orders WHERE order_id <= 3")
spark.sql("UPDATE audit_demo SET status = 'cancelled' WHERE order_id = 1")
spark.sql("DELETE FROM audit_demo WHERE order_id = 3")
spark.sql("DESCRIBE HISTORY audit_demo").select("version", "operation").orderBy("version").show(truncate=False)   # a command, not a subquery
show("SELECT 'v0' AS version, * FROM audit_demo VERSION AS OF 0 UNION ALL SELECT 'now', * FROM audit_demo ORDER BY version DESC, order_id")
spark.sql("RESTORE TABLE audit_demo TO VERSION AS OF 0")
show("SELECT * FROM audit_demo ORDER BY order_id")

**Next:** practise these entries in [SQL practice 10: changing data: DML, MERGE and history](sql-practice-10-changing-data.ipynb).